# Interactive Base vs SFT vs DPO — Streamlit demo
Two tabs: **Base vs SFT vs DPO** (the full three-way comparison) and
**Base vs SFT** (just the fine-tuning step in isolation). Pick one of the
10 fixed test questions in either tab, hit **Get answers**, and see the
models' answers side by side — the winning one is badged, with the actual
reason it won pulled straight from your own verified `reports/*.md` files.
Each tab has its own live win-tally, donut chart, and bar chart.


In [34]:
# mount Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'

REQUIRED_REPORTS = ['reports/final_evaluation.md', 'reports/sft_model_comparison.md']
missing = [p for p in REQUIRED_REPORTS if not os.path.exists(f'{PROJECT}/{p}')]
if missing:
    raise FileNotFoundError(
        f"Missing: {missing} -- run notebooks/instruction_finetuning.ipynb (Stage 2) "
        "and notebooks/dpo_alignment.ipynb (Stage 3) all the way through first, "
        "this demo needs both reports."
    )
print('Found both reports:', REQUIRED_REPORTS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found both reports: ['reports/final_evaluation.md', 'reports/sft_model_comparison.md']


In [35]:
!pip install -q streamlit
!which streamlit

/usr/local/bin/streamlit


##App

In [36]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="Base vs SFT vs DPO", page_icon=":material/quiz:", layout="wide")

PROJECT = "/content/drive/MyDrive/Domain-SupportSpecialist"

# the same 10 fixed questions used across every notebook in this project --
# used as anchors so the parser below doesn't get confused by answers that
# span multiple physical lines in the markdown file.
QUESTIONS = [
    "How can I cancel my order after it has been placed?",
    "My package says delivered but I never received it, what do I do?",
    "How long does a refund take to appear on my card?",
    "Can I change the delivery address after checkout?",
    "The item I received is damaged, what are my options?",
    "How do I track my order status?",
    "I was charged twice for one order, how do I fix this?",
    "Can I get a replacement instead of a refund?",
    "What happens if I miss the delivery attempt?",
    "How do I apply a discount code after placing an order?",
]

CHART_BG = "#0e1117"  # matches Streamlit's own dark theme background


@st.cache_data
def load_report(filename, header):
    path = f"{PROJECT}/reports/{filename}"
    with open(path, encoding="utf-8") as f:
        text = f.read()

    # find where each question's row starts -- robust to multi-line cell
    # content (numbered steps etc.), unlike a naive line-by-line parser
    anchors = []
    for q in QUESTIONS:
        marker = f"| {q} |"
        idx = text.find(marker)
        if idx != -1:
            anchors.append((idx, q))
    anchors.sort()

    rows = []
    for i, (idx, q) in enumerate(anchors):
        end = anchors[i + 1][0] if i + 1 < len(anchors) else len(text)
        block = text[idx:end].strip()
        parts = [p.strip() for p in block.split("|")]
        parts = [p for p in parts if p != ""] if parts and parts[0] == "" else parts
        if len(parts) >= len(header):
            rows.append(dict(zip(header, parts[:len(header)])))
    return rows


st.title("Customer Support Assistant")
st.caption(
    "Answers shown here are pre-computed from your own verified evaluation reports -- "
    "this demo doesn't run the models live."
)


def render_comparison(state_prefix, rows, models, model_colors, verdict_col, reason_col, labels):
    """One self-contained comparison panel -- KPI row, donut+bar chart, question
    picker, side-by-side answers. Used for both the 3-way and 2-way tabs below,
    each with its own session-state keys (state_prefix) so they don't collide."""
    by_question = {r["Question"]: r for r in rows}
    asked_key, tally_key, last_key = f"{state_prefix}_asked", f"{state_prefix}_tally", f"{state_prefix}_last"

    if asked_key not in st.session_state:
        st.session_state[asked_key] = set()
    if tally_key not in st.session_state:
        st.session_state[tally_key] = {m: 0 for m in models}
    if last_key not in st.session_state:
        st.session_state[last_key] = None

    # ---- KPI row: live, updates as you ask more questions ----
    with st.container(horizontal=True):
        for m in models:
            st.metric(f"{m} wins", st.session_state[tally_key][m], border=True)
        st.metric("Questions asked", f"{len(st.session_state[asked_key])} / {len(rows)}", border=True)

    # ---- donut + bar chart, both reflect the same live tally ----
    # drawn with matplotlib (st.pyplot) instead of st.vega_lite_chart / st.bar_chart --
    # those two kept either silently failing to repaint on rerun or breaking on a
    # `key` argument the installed Streamlit version's bar_chart doesn't support.
    # A fresh matplotlib figure is rebuilt from scratch every rerun, so nothing goes stale.
    wins = [st.session_state[tally_key][m] for m in models]
    chart_col1, chart_col2 = st.columns(2)
    with chart_col1:
        with st.container(border=True):
            st.markdown("**Win share**")
            if sum(wins) > 0:
                fig, ax = plt.subplots(figsize=(2.6, 2.6), dpi=150)
                fig.patch.set_facecolor(CHART_BG)
                ax.set_facecolor(CHART_BG)
                nonzero = [(m, w, c) for m, w, c in zip(models, wins, model_colors) if w > 0]
                ax.pie(
                    [w for _, w, _ in nonzero],
                    labels=[m for m, _, _ in nonzero],
                    colors=[c for _, _, c in nonzero],
                    autopct=lambda pct: f"{round(pct * sum(wins) / 100)}",
                    wedgeprops={"width": 0.45},
                    startangle=90,
                    textprops={"fontsize": 8, "color": "white"},
                )
                ax.set_aspect("equal")
                fig.tight_layout()
                # use_container_width=False -- otherwise Streamlit re-stretches this
                # small figure back out to fill the full column width regardless of figsize
                st.pyplot(fig, clear_figure=True, use_container_width=False)
            else:
                st.caption("Ask a question below to start filling this in.")
    with chart_col2:
        with st.container(border=True):
            st.markdown("**Win count**")
            fig, ax = plt.subplots(figsize=(2.6, 2.6), dpi=150)
            fig.patch.set_facecolor(CHART_BG)
            ax.set_facecolor(CHART_BG)
            ax.bar(models, wins, color=model_colors)
            ax.set_ylabel("Wins", fontsize=8, color="white")
            ax.set_ylim(0, max(wins + [1]) + 1)
            ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
            ax.tick_params(labelsize=8, colors="white")
            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)
            for spine in ["left", "bottom"]:
                ax.spines[spine].set_color("white")
            fig.tight_layout()
            st.pyplot(fig, clear_figure=True, use_container_width=False)

    st.divider()

    # ---- Question picker -- keys are prefixed so the two tabs' widgets don't collide ----
    question = st.selectbox(
        "Pick a test question", options=list(by_question.keys()), key=f"{state_prefix}_select"
    )
    show = st.button(
        "Get answers", type="primary", icon=":material/send:", key=f"{state_prefix}_button"
    )

    if show:
        if question not in st.session_state[asked_key]:
            st.session_state[asked_key].add(question)
            winner = by_question[question][verdict_col]
            st.session_state[tally_key][winner] = st.session_state[tally_key].get(winner, 0) + 1
        st.session_state[last_key] = question

    if st.session_state[last_key]:
        row = by_question[st.session_state[last_key]]
        winner = row[verdict_col]

        cols = st.columns(len(models), border=True)
        for col, m in zip(cols, models):
            with col:
                if m == winner:
                    st.markdown(f"**{labels[m]}** :green-badge[:material/trophy: Best answer]")
                else:
                    st.markdown(f"**{labels[m]}** :red-badge[Not the best here]")
                st.write(row[m])

        st.subheader(f":material/lightbulb: Why {labels[winner]} wins here", divider="gray")
        st.info(row[reason_col], icon=":material/info:")


tab_three, tab_two = st.tabs(["Base vs SFT vs DPO", "Base vs SFT"])

with tab_three:
    rows_three = load_report(
        "final_evaluation.md", ["Question", "Base", "SFT", "DPO", "Best Answer", "Reason"]
    )
    st.caption(f"{len(rows_three)} fixed test questions loaded, three models, one honest verdict each.")
    render_comparison(
        state_prefix="three",
        rows=rows_three,
        models=["Base", "SFT", "DPO"],
        #model_colors=["#46a0fa", "#53fc6a", "#fc5bc1"],
        model_colors=["#4494fc", "#b7f777", "#fc81d1"],
        verdict_col="Best Answer",
        reason_col="Reason",
        labels={
            "Base": "Base model (untrained)",
            "SFT": "SFT (instruction fine-tuned)",
            "DPO": "DPO (final model)",
        },
    )

with tab_two:
    rows_two = load_report(
        "sft_model_comparison.md", ["Question", "Base", "SFT", "Which is Better?", "Reason"]
    )
    st.caption(f"{len(rows_two)} fixed test questions loaded, Base vs the instruction-tuned model.")
    render_comparison(
        state_prefix="two",
        rows=rows_two,
        models=["Base", "SFT"],
        model_colors=["#CBD5E1", "#AEE1FF"],
        verdict_col="Which is Better?",
        reason_col="Reason",
        labels={"Base": "Base model (untrained)", "SFT": "SFT (instruction fine-tuned)"},
    )


Overwriting app.py


## Launch it


In [37]:
import subprocess, time, urllib.request

# kill any previous instance still running from an earlier cell run
subprocess.run(['pkill', '-f', 'streamlit run'], capture_output=True)
time.sleep(1)

log_file = open('streamlit.log', 'w')
proc = subprocess.Popen(
    [
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true',
        # required for the app to actually render (not just load a blank shell)
        # when accessed through Colab's proxied iframe instead of a normal browser tab
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false',
    ],
    stdout=log_file, stderr=subprocess.STDOUT,
)

# actively wait until the server is really answering, instead of a blind sleep
for attempt in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen('http://localhost:8501', timeout=1)
        print(f'Streamlit is up after {attempt+1}s (pid {proc.pid}).')

        break
    except Exception:
        if proc.poll() is not None:
            print('Streamlit process exited early -- see the log below:')
            print(open('streamlit.log').read())
            break
else:
    print('Streamlit did not respond in time -- run the diagnostic cell below to see streamlit.log for errors.')

Streamlit is up after 3s (pid 14741).


In [38]:
# diagnostic -- run this any time to check what's actually going on
print("=== is streamlit installed? ===")
!which streamlit

print("\n=== does app.py exist? ===")
!ls -la app.py

print("\n=== is anything running on port 8501? ===")
!ps aux | grep streamlit

# the log's own Local/Network/External URL lines are filtered out here on
# purpose -- they're addresses inside the Colab VM, not clickable from your
# browser, and only cause confusion. The iframe cell below is the real view.
print("\n=== streamlit.log contents (URL lines hidden -- see note above) ===")
!grep -v "URL:" streamlit.log 2>&1 || echo "(log file doesn't exist yet)"

=== is streamlit installed? ===
/usr/local/bin/streamlit

=== does app.py exist? ===
-rw-r--r-- 1 root root 8554 Jul 12 08:56 app.py

=== is anything running on port 8501? ===
root       14741 58.0  0.5 525728 71652 ?        Sl   08:56   0:02 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false
root       14778  0.0  0.0   7372  3388 ?        S    08:56   0:00 /bin/bash -c ps aux | grep streamlit
root       14780  0.0  0.0   6480  2392 ?        S    08:56   0:00 grep streamlit

=== streamlit.log contents (URL lines hidden -- see note above) ===


2026-07-12 08:56:38.222 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.




In [39]:
from google.colab.output import eval_js
from IPython.display import display, HTML

proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
display(HTML(
    f'<a href="{proxy_url}" target="_blank" '
    f'style="font-size:16px">Open Streamlit app in a new tab &#8599;</a>'
))
print("If the link above doesn't render, copy this URL manually:", proxy_url)

If the link above doesn't render, copy this URL manually: https://8501-m-s-kkb-usw1c2-1fnd3jewmfwlz-c.us-west1-2.prod.colab.dev


## Stopping the app
Run this when you're done, or just disconnect the runtime -- it'll clean up
on its own either way.

In [40]:
# import subprocess
# subprocess.run(['pkill', '-f', 'streamlit run'], capture_output=True)
# print('Stopped.')